In [ ]:
# Install dependencies
!pip install -q torch torchvision scikit-learn scikit-image opencv-python tqdm pandas pillow

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print('\n✓ Setup complete!')

In [ ]:
# Step 2: Upload Data and Code via Google Drive (FAST!)
# This is much faster than direct upload for large files (3.4GB)

from google.colab import drive
import tarfile
import zipfile
import os

# Mount Google Drive
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')
print("✅ Drive mounted!\n")

print("📋 INSTRUCTIONS:")
print("1. Upload data.tar.gz and code.zip to your Google Drive")
print("2. Update the paths below to match your Drive folder\n")

# Update these paths to match where you uploaded files in Google Drive
drive_data_path = '/content/drive/MyDrive/data.tar.gz'  # or data.zip
drive_code_path = '/content/drive/MyDrive/code.zip'

# Extract data (supports both .tar.gz and .zip)
print("📦 Extracting data from Drive...")
if drive_data_path.endswith('.tar.gz'):
    with tarfile.open(drive_data_path, 'r:gz') as tar_ref:
        tar_ref.extractall('.')
else:
    with zipfile.ZipFile(drive_data_path, 'r') as zip_ref:
        zip_ref.extractall('.')
print("✅ Data extracted")

# Extract code
print("\n📦 Extracting code...")
with zipfile.ZipFile(drive_code_path, 'r') as zip_ref:
    zip_ref.extractall('.')
print("✅ Code extracted\n✅ All files ready!")

In [ ]:
# Verify data counts
import os

def count_images(directory):
    classes = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg'))])
            classes[class_name] = count
    return classes

print('Training Data:')
train_counts = count_images('data/train')
for cls, count in sorted(train_counts.items()):
    print(f'  {cls}: {count}')

print('\nValidation Data:')
val_counts = count_images('data/val')
for cls, count in sorted(val_counts.items()):
    print(f'  {cls}: {count}')

print(f'\nTotal Training: {sum(train_counts.values())}')
print(f'Total Validation: {sum(val_counts.values())}')
print('\n✓ Data verified!')

In [ ]:
# GPU-OPTIMIZED TRAINING with Automatic Mixed Precision (AMP)
print('🚀 GPU Training: Mixed Precision + Larger Batch Size')
print('Target: 75-85% Accuracy with balanced class detection')
print('Time: ~30-45 minutes (vs 2+ hours on CPU)')
print('=' * 80)

!cd /content && python backend/train_hybrid.py \
    --data-dir data \
    --epochs 100 \
    --batch-size 64 \
    --learning-rate 0.0003 \
    --early-stopping-patience 25 \
    --num-workers 4 \
    --checkpoint-dir checkpoints

In [ ]:
# Check training results
import os

print('Training Complete!\n')
print('Generated checkpoints:')
!ls -lh checkpoints/*.pth

# Show best model info
if os.path.exists('checkpoints/best_hybrid_model.pth'):
    import torch
    checkpoint = torch.load('checkpoints/best_hybrid_model.pth', map_location='cpu', weights_only=False)
    print(f"\n✓ Best Model:")
    print(f"  Validation Accuracy: {checkpoint.get('val_accuracy', 'N/A'):.2f}%")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Classes: {checkpoint.get('num_classes', 'N/A')}")
else:
    print('\n⚠ Best model not found!')

In [ ]:
# Download trained model
from google.colab import files
import os

# Zip all checkpoints
!zip -r trained_model.zip checkpoints/

print('Downloading trained model...')
files.download('trained_model.zip')

print('\n✓ Download started!')
print('Extract on your Mac and copy to:')
print('/Users/akshitarora/cervical-cancer-classifier-local/backend/checkpoints/')

---
## Next Steps on Your Mac:

```bash
cd ~/Downloads
unzip trained_model.zip

# Copy to your project
cp -r checkpoints/* /Users/akshitarora/cervical-cancer-classifier-local/backend/checkpoints/

# Your trained model is ready to use!
```

## Expected Results with GPU Optimization:
- **Validation Accuracy**: 75-85% (improved with GPU training)
- **Training Time**: 30-45 minutes (3-4x faster than CPU)
- **Model Size**: ~2.2 MB
- **GPU Features Enabled**: 
  - ✓ Automatic Mixed Precision (AMP) for 2x speed
  - ✓ Batch size 64 (vs 32 on CPU)
  - ✓ Learning rate 0.0006 (2x base for GPU)
  - ✓ 4 data workers with persistent workers
  - ✓ Pin memory + prefetch for faster loading
- **All classes balanced**: Better minority class detection